# 04 — Analysis, Significance Tests, and Report Figures

Loads all pre-computed metrics, runs paired t-tests with Holm-Bonferroni
correction, and produces four final report figures.

**Prerequisites** (run notebooks 01 and 03 first):
- `results/metrics/bm25.json`
- `results/metrics/colbert_nbits1.json`
- `results/metrics/colbert_nbits2.json`
- `results/metrics/colbert_nbits4.json`
- `results/metrics/ablation_nbits.csv`

In [1]:
import os, sys

# ── Auto-detect environment ───────────────────────────────────────────────────
try:
    import google.colab; _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_IN_KAGGLE = os.path.exists("/kaggle/working")

if _IN_COLAB or _IN_KAGGLE:
    # Remote: repo will be cloned in the next cell — update REPO_URL first.
    REPO_URL  = "https://github.com/YOUR_ORG/cs410-tech-review.git"  # ← REPLACE
    REPO_ROOT = "/content/cs410-tech-review" if _IN_COLAB else "/kaggle/working/cs410-tech-review"
else:
    # Local: locate the repo root relative to this notebook.
    _here = os.path.abspath(".")
    if os.path.basename(_here) == "notebooks" and os.path.isdir(os.path.join(_here, "..", "src")):
        REPO_ROOT = os.path.abspath(os.path.join(_here, ".."))
    elif os.path.isdir(os.path.join(_here, "src")):
        REPO_ROOT = _here
    else:
        REPO_ROOT = _here  # fallback: set manually if this prints the wrong path
    REPO_URL = None
    print(f"Local mode — REPO_ROOT: {REPO_ROOT}")

Local mode — REPO_ROOT: /home/kaiyul3/cs410-tech-review


In [2]:
# Clone repo (Colab / Kaggle only — skipped automatically in local mode).
if REPO_URL and not os.path.isdir(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
elif REPO_URL:
    print(f"Repo already present at {REPO_ROOT}")
else:
    print("Local mode — skipping clone.")

Local mode — skipping clone.


In [3]:
if _IN_COLAB or _IN_KAGGLE:
    %pip install -q \
        "torch>=2.1.0,<2.4.0" \
        "transformers==4.44.2" \
        "tokenizers<0.20" \
        "faiss-cpu>=1.7.4" \
        "ragatouille==0.0.9.post2" \
        "colbert-ai>=0.2.19" \
        "langchain==0.1.20" \
        "langchain-core==0.1.53" \
        "rank_bm25>=0.2.2" \
        "beir>=2.0.0" \
        "ranx>=0.3.16" \
        "scipy>=1.11.0" \
        "numpy>=1.24.0,<2.0.0" \
        "pandas>=2.0.0" \
        "pydantic>=2.0.0" \
        "matplotlib>=3.7.0" \
        "seaborn>=0.12.0" \
        "tqdm>=4.65.0" \
        "pyyaml>=6.0" \
        "ninja"
    print("Dependencies installed.")
else:
    # Local: colbert-review conda env already provides everything.
    # Inject the env's bin dir at the head of PATH so subprocesses the kernel
    # spawns (notably ColBERT's JIT C++ extension build via ninja) resolve the
    # right ninja/g++/nvcc, regardless of how the kernel was launched.
    import sys
    _env_bin = os.path.dirname(sys.executable)
    _path = os.environ.get("PATH", "")
    if _env_bin not in _path.split(os.pathsep):
        os.environ["PATH"] = _env_bin + os.pathsep + _path
    print(f"Local mode — using existing env at {_env_bin}")

Local mode — using existing env at /home/kaiyul3/.conda/envs/colbert-review/bin


In [4]:
import sys, os, json
from pathlib import Path

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

RUNS_DIR    = Path(REPO_ROOT) / "results" / "runs"
METRICS_DIR = Path(REPO_ROOT) / "results" / "metrics"
FIGURES_DIR = Path(REPO_ROOT) / "results" / "figures"

for d in [RUNS_DIR, METRICS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)
print("Paths ready.")

Paths ready.


## 0. Verify prerequisites

In [5]:
REQUIRED = [
    METRICS_DIR / "bm25.json",
    METRICS_DIR / "colbert_nbits1.json",
    METRICS_DIR / "colbert_nbits2.json",
    METRICS_DIR / "colbert_nbits4.json",
    METRICS_DIR / "ablation_nbits.csv",
]
missing = [str(p) for p in REQUIRED if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prerequisite files — run notebooks 01 and 03 first:\n"
        + "\n".join(missing)
    )
print("All prerequisite files present.")

FileNotFoundError: Missing prerequisite files — run notebooks 01 and 03 first:
/home/kaiyul3/cs410-tech-review/results/metrics/colbert_nbits1.json

## 1. Load all evaluation results

In [ ]:
import pandas as pd

def _load(path):
    with open(path) as f:
        return json.load(f)

SYSTEMS = ["bm25", "colbert_nbits1", "colbert_nbits2", "colbert_nbits4"]
LABELS  = ["BM25", "ColBERT nbits=1", "ColBERT nbits=2", "ColBERT nbits=4"]

all_evals = {
    "bm25":           _load(METRICS_DIR / "bm25.json"),
    "colbert_nbits1": _load(METRICS_DIR / "colbert_nbits1.json"),
    "colbert_nbits2": _load(METRICS_DIR / "colbert_nbits2.json"),
    "colbert_nbits4": _load(METRICS_DIR / "colbert_nbits4.json"),
}

agg_rows = [{"system": name, **data["metrics"]} for name, data in all_evals.items()]
agg_df   = pd.DataFrame(agg_rows).set_index("system")
print("=== Aggregate metrics ===")
display(agg_df.style.format("{:.4f}").highlight_max(axis=0, color="#d4edda"))

## 2. Paired t-tests — BM25 vs. each ColBERT variant

Delta = ColBERT − BM25 (positive means ColBERT wins).

In [ ]:
from src.eval.stats import paired_ttest, holm_bonferroni

COMPARISONS    = [("colbert_nbits1","ColBERT nbits=1"),
                  ("colbert_nbits2","ColBERT nbits=2"),
                  ("colbert_nbits4","ColBERT nbits=4")]
TARGET_METRICS = ["ndcg@10", "map"]

results = []
for sys_key, sys_label in COMPARISONS:
    for metric in TARGET_METRICS:
        results.append(paired_ttest(
            a=all_evals["bm25"]["per_query"],
            b=all_evals[sys_key]["per_query"],
            metric=metric, system_a="BM25", system_b=sys_label,
            n_bootstrap=2000, seed=42,
        ))

print(f"{'System B':<22} {'Metric':<12} {'mean_A':>8} {'mean_B':>8} "
      f"{'delta':>8} {'p-value':>9} {'95% CI':>22}")
print("-" * 95)
for r in results:
    ci = f"[{r.ci_low:+.4f}, {r.ci_high:+.4f}]"
    print(f"{r.system_b:<22} {r.metric:<12} {r.mean_a:>8.4f} {r.mean_b:>8.4f} "
          f"{r.mean_delta:>+8.4f} {r.p_value:>9.4f} {ci:>22}")

## 3. Holm-Bonferroni correction

In [ ]:
decisions = holm_bonferroni([r.p_value for r in results], alpha=0.05)

print("=== Holm-Bonferroni corrected (alpha=0.05) ===\n")
print(f"{'System B':<22} {'Metric':<12} {'p-value':>9} {'sig?':>6}")
print("-" * 55)
for r, sig in zip(results, decisions):
    print(f"{r.system_b:<22} {r.metric:<12} {r.p_value:>9.4f} "
          f"{'YES *' if sig else 'no':>6}")

## 4. Pairwise ablation comparisons within ColBERT

In [ ]:
ABLATION_PAIRS = [
    ("colbert_nbits1","colbert_nbits2","nbits1 vs nbits2"),
    ("colbert_nbits2","colbert_nbits4","nbits2 vs nbits4"),
    ("colbert_nbits1","colbert_nbits4","nbits1 vs nbits4"),
]

abl_results  = [
    paired_ttest(
        a=all_evals[ka]["per_query"], b=all_evals[kb]["per_query"],
        metric="ndcg@10", system_a=ka, system_b=kb, n_bootstrap=2000, seed=42,
    )
    for ka, kb, _ in ABLATION_PAIRS
]
abl_decisions = holm_bonferroni([r.p_value for r in abl_results], alpha=0.05)

print("=== Pairwise ablation — NDCG@10 (Holm-Bonferroni) ===\n")
print(f"{'Comparison':<25} {'delta':>8} {'p-value':>9} {'sig?':>6} {'95% CI':>22}")
print("-" * 75)
for (_, _, label), r, sig in zip(ABLATION_PAIRS, abl_results, abl_decisions):
    ci = f"[{r.ci_low:+.4f}, {r.ci_high:+.4f}]"
    print(f"{label:<25} {r.mean_delta:>+8.4f} {r.p_value:>9.4f} "
          f"{'YES *' if sig else 'no':>6} {ci:>22}")

## 5. Figure 1 — System comparison bar chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PALETTE = ["#5C85D6","#E8505B","#F9A825","#43A047"]
MPLOT   = ["ndcg@10","mrr@10","map","recall@100"]
x, width = np.arange(len(MPLOT)), 0.18

fig, ax = plt.subplots(figsize=(10, 5))
for i, (sk, lb, col) in enumerate(zip(SYSTEMS, LABELS, PALETTE)):
    vals = [all_evals[sk]["metrics"].get(m, 0.0) for m in MPLOT]
    ax.bar(x + (i - 1.5) * width, vals, width,
           label=lb, color=col, alpha=0.88, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(MPLOT, fontsize=11)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("BM25 vs. ColBERT (nbits ablation) — SciFact test")
ax.legend(fontsize=9, ncol=4)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure1_system_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figure1_system_comparison.png")

## 6. Figure 2 — Ablation size-quality Pareto

In [ ]:
abl_df = pd.read_csv(METRICS_DIR / "ablation_nbits.csv")
abl_df = abl_df.rename(columns=lambda c: c.replace("metric_","") if c.startswith("metric_") else c)

NB_COLORS = {1:"#E8505B", 2:"#F9A825", 4:"#43A047"}

fig, ax = plt.subplots(figsize=(7, 5))
for _, row in abl_df.iterrows():
    nb = int(row["nbits"])
    ax.scatter(row["index_mib"], row["ndcg@10"],
               s=160, color=NB_COLORS.get(nb,"grey"), zorder=4)
    ax.annotate(f"  nbits={nb} ({row['ndcg@10']:.3f})",
                xy=(row["index_mib"], row["ndcg@10"]), fontsize=9, va="center")

bm25_ndcg = all_evals["bm25"]["metrics"].get("ndcg@10")
if bm25_ndcg:
    ax.axhline(bm25_ndcg, color="steelblue", linestyle="--", linewidth=1.5,
               label=f"BM25 NDCG@10 = {bm25_ndcg:.3f}")

ax.set_xlabel("Index size (MiB)")
ax.set_ylabel("NDCG@10")
ax.set_title("ColBERT residual compression: size vs. quality")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure2_ablation_pareto.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figure2_ablation_pareto.png")

## 7. Figure 3 — Significance heatmap

In [ ]:
import seaborn as sns

SIG_METRICS = ["ndcg@10", "map"]
SYS_KEYS    = ["colbert_nbits1","colbert_nbits2","colbert_nbits4"]
SYS_LBLS    = ["ColBERT nbits=1","ColBERT nbits=2","ColBERT nbits=4"]

delta_mat  = np.zeros((len(SYS_KEYS), len(SIG_METRICS)))
annot_mat  = np.empty_like(delta_mat, dtype=object)

for i, sk in enumerate(SYS_KEYS):
    for j, m in enumerate(SIG_METRICS):
        r = paired_ttest(
            a=all_evals["bm25"]["per_query"],
            b=all_evals[sk]["per_query"],
            metric=m, system_a="BM25", system_b=sk, n_bootstrap=2000, seed=42,
        )
        delta_mat[i, j] = r.mean_delta
        annot_mat[i, j] = f"{r.mean_delta:+.3f}" + (" *" if r.p_value < 0.05 else "")

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(delta_mat, annot=annot_mat, fmt="",
            xticklabels=SIG_METRICS, yticklabels=SYS_LBLS,
            center=0, cmap="RdYlGn", linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "delta (ColBERT - BM25)"})
ax.set_title("Mean delta vs. BM25 (* p<0.05, uncorrected)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure3_significance_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figure3_significance_heatmap.png")

## 8. Figure 4 — Per-query NDCG@10 box plots

In [ ]:
PER_Q_DATA = {
    "BM25":              [v.get("ndcg@10",0.0) for v in all_evals["bm25"]["per_query"].values()],
    "ColBERT\nnbits=1": [v.get("ndcg@10",0.0) for v in all_evals["colbert_nbits1"]["per_query"].values()],
    "ColBERT\nnbits=2": [v.get("ndcg@10",0.0) for v in all_evals["colbert_nbits2"]["per_query"].values()],
    "ColBERT\nnbits=4": [v.get("ndcg@10",0.0) for v in all_evals["colbert_nbits4"]["per_query"].values()],
}
BOX_COLORS = ["#5C85D6","#E8505B","#F9A825","#43A047"]

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(list(PER_Q_DATA.values()), labels=list(PER_Q_DATA.keys()),
                patch_artist=True, notch=True,
                medianprops=dict(color="black", linewidth=2))
for patch, c in zip(bp["boxes"], BOX_COLORS):
    patch.set_facecolor(c)
    patch.set_alpha(0.75)

ax.set_ylabel("NDCG@10")
ax.set_title("Per-query NDCG@10 distribution — SciFact test")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure4_perquery_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figure4_perquery_boxplot.png")

## 9. Final metrics table

In [ ]:
summary = [
    {"System": lb,
     **{m: f"{all_evals[sk]['metrics'].get(m, float('nan')):.4f}"
        for m in ["ndcg@10","mrr@10","map","recall@100"]}}
    for sk, lb in zip(SYSTEMS, LABELS)
]
summary_df = pd.DataFrame(summary).set_index("System")
display(summary_df)

table_path = FIGURES_DIR / "final_metrics_table.csv"
summary_df.to_csv(table_path)
print(f"Saved: {table_path}")

## Findings template

Fill this in before submitting:

**RQ1 — Does ColBERT outperform BM25 on SciFact?**
- ColBERT nbits=2 NDCG@10: ___ | BM25 NDCG@10: ___
- p-value: ___ | Holm-Bonferroni corrected: ___

**RQ2 — Trade-off across nbits ∈ {1, 2, 4}?**
- Index size range: ___ — ___ MiB
- NDCG@10 range: ___ — ___
- Pairwise significance: ___